In [1]:
import requests
import json
import pandas as pd
from datetime import datetime
import ast
import operator
import re

# --- 1. Mongol Bank Function ---
def mongol_bank_khansh(date: str) -> pd.DataFrame:
    # 1. Validate date format (YYYY-MM-DD)
    try:
        datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        raise ValueError("Invalid date format. Expected YYYY-MM-DD")

    url = f"https://www.mongolbank.mn/mn/currency-rates/data?startDate={date}&endDate={date}"

    try:
        response = requests.post(url)
    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Request failed: {e}")

    # 2. Check status code
    if 400 <= response.status_code < 500:
        raise Exception(f"Client error ({response.status_code}): Check request parameters")
    elif 500 <= response.status_code < 600:
        raise Exception(f"Server error ({response.status_code}): MongolBank API issue")

    if response.status_code != 200:
        raise Exception(f"Unexpected status code: {response.status_code}")

    # 3. Validate JSON response
    try:
        data = response.json()
    except ValueError:
        return "API дуудхад алдаа гарлаа, хөгжүүлэгчтэй холбогдоорой"

    if not data.get("success", False):
        return "API дуудхад алдаа гарлаа. Түр хүлээгээд дахиад асуугаарай"

    if "data" not in data:
        return "Тухайн өдрийн ханшийн мэдээлэл байхгүй байна"

    return pd.DataFrame(data["data"])

# --- 2. Math Evaluator ---
OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg
}

def eval_expr(expr: str):
    def _eval(node):
        if isinstance(node, ast.Num):
            return node.n
        
        elif isinstance(node, ast.BinOp):
            if type(node.op) not in OPS:
                return "Unsupported operator"
            return OPS[type(node.op)](_eval(node.left), _eval(node.right))
        
        elif isinstance(node, ast.UnaryOp):
            if type(node.op) not in OPS:
                return "Unsupported unary operator"
            return OPS[type(node.op)](_eval(node.operand))
        
        else:
            return "Unsupported expression"

    try:
        parsed = ast.parse(expr, mode='eval')
        result = _eval(parsed.body)

        if isinstance(result, str):
            return result

        return result

    except ZeroDivisionError:
        return "Division by zero"
    except Exception:
        return "Invalid expression"

# --- 3. User Query Handler ---
def handle_user_query(user_query: str):
    if not isinstance(user_query, str):
        return "Invalid input"

    query = user_query.strip()
    query_lower = query.lower()

    # 1. Contact шалгах
    if any(word in query_lower for word in ["утас", "contact", "холбоо барих", "holbodgoh medeelel"]):
        return "Манай холбоо барих утасны дугаар: 99887766"

    # 2. Location шалгах
    if any(word in query_lower for word in ["байршил", "location"]):
        return "Манай байршил: Galaxy tower 7 давхар, 705 тоот"

    # 3. Date format шалгах (YYYY-MM-DD)
    date_pattern = r"^\d{4}-\d{2}-\d{2}$"
    if re.match(date_pattern, query):
        try:
            return mongol_bank_khansh(query)
        except Exception as e:
            return str(e)

    # 4. Math expression гэж үзэх оролдлого
    math_pattern = r"^[\d+\-*/().\s^]+$"
    if re.match(math_pattern, query):
        try:
            return eval_expr(query)
        except Exception as e:
            return str(e)

    # 5. Default
    return user_query

# --- Testing ---
if __name__ == "__main__":
    # Жишээ тестүүд
    print(handle_user_query("2026-01-01"))
    print(handle_user_query("location"))
    print(handle_user_query("(2+4)*5"))

    RATE_DATE       USD       EUR    JPY       CHF     SEK       GBP  \
0  2026-01-01  3,556.81  4,169.83  22.71  4,478.76  385.82  4,778.40   

        BGN    HUF    EGP  ...     PLN    UAH     NOK    NPR     ZAR    TRY  \
0  2,138.98  10.83  74.65  ...  987.45  83.98  353.11  24.74  214.29  82.79   

    VND            XAU         XAG       SDR  
0  0.14  15,348,115.32  255,887.94  4,875.36  

[1 rows x 39 columns]
Манай байршил: Galaxy tower 7 давхар, 705 тоот
30


C:\Users\oyundelger.ch\AppData\Local\Temp\ipykernel_28412\261395111.py:59: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):
C:\Users\oyundelger.ch\AppData\Local\Temp\ipykernel_28412\261395111.py:60: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  return node.n
